In [1]:
from analytics.db import query

In [2]:
query("""SELECT * FROM sellers LIMIT 5  """)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [3]:
query("""SELECT * FROM order_items LIMIT 5  """)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [4]:
query("""SELECT
    seller_state,
    COUNT(*) AS sellers_count
FROM sellers
GROUP BY seller_state
HAVING COUNT(*) >= 50
ORDER BY sellers_count DESC;
""")

,seller_state,sellers_count
0,SP,1849
1,PR,349
2,MG,244
3,SC,190
4,RJ,171
5,RS,129


In [5]:
query("""
    SELECT order_id, COUNT(*) AS items, SUM(price) AS items_total
    FROM order_items
    WHERE order_id = '06d5cbe68295558fd6c48998db3ea397'
    GROUP BY order_id
""")

,order_id,items,items_total
0,06d5cbe68295558fd6c48998db3ea397,3,123.0


In [6]:
query("""
    SELECT
        order_id,
        order_item_id,
        price,
        COUNT(*)   OVER (PARTITION BY order_id) AS items,
        SUM(price) OVER (PARTITION BY order_id) AS items_total
    FROM order_items
    WHERE order_id = '06d5cbe68295558fd6c48998db3ea397'
    ORDER BY order_item_id
""")

,order_id,order_item_id,price,items,items_total
0,06d5cbe68295558fd6c48998db3ea397,1,39.0,3,123.0
1,06d5cbe68295558fd6c48998db3ea397,2,33.0,3,123.0
2,06d5cbe68295558fd6c48998db3ea397,3,51.0,3,123.0


In [7]:
query("""
    SELECT
        order_id,
        order_item_id,
        price,
        COUNT(*)   OVER (PARTITION BY order_id) AS items,
        SUM(price) OVER (PARTITION BY order_id) AS items_total,
        ROUND(100 * price / SUM(price) OVER (PARTITION BY order_id), 1) AS pct_of_order
    FROM order_items
    WHERE order_id = '06d5cbe68295558fd6c48998db3ea397'
    ORDER BY order_item_id
""")

,order_id,order_item_id,price,items,items_total,pct_of_order
0,06d5cbe68295558fd6c48998db3ea397,1,39.0,3,123.0,31.7
1,06d5cbe68295558fd6c48998db3ea397,2,33.0,3,123.0,26.8
2,06d5cbe68295558fd6c48998db3ea397,3,51.0,3,123.0,41.5


In [8]:
query("""
    SELECT
        c.customer_state,
        COUNT(*) AS orders,
        SUM(COUNT(*)) OVER () AS all_orders,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_all
    FROM orders AS o
    JOIN customers AS c ON c.customer_id = o.customer_id
    GROUP BY c.customer_state
    ORDER BY orders DESC
""")

,customer_state,orders,all_orders,pct_of_all
0,SP,41746,99441.0,42.0
1,RJ,12852,99441.0,12.9
2,MG,11635,99441.0,11.7
3,RS,5466,99441.0,5.5
4,PR,5045,99441.0,5.1
5,SC,3637,99441.0,3.7
6,BA,3380,99441.0,3.4
7,DF,2140,99441.0,2.2
8,ES,2033,99441.0,2.0
9,GO,2020,99441.0,2.0


In [9]:
per_state_category = """
    per_state_category AS (
        -- one row = one category in one customer state
        SELECT
            c.customer_state,
            COALESCE(t.product_category_name_english, 'unknown') AS category,
            COUNT(*) AS items
        FROM order_items AS i
        JOIN orders AS o ON o.order_id = i.order_id
        JOIN customers AS c ON c.customer_id = o.customer_id
        JOIN products AS p ON p.product_id = i.product_id
        LEFT JOIN category_translation AS t ON t.product_category_name = p.product_category_name
        GROUP BY c.customer_state, category
    )
"""

In [10]:
query(f"""
    WITH {per_state_category}
    SELECT
        customer_state,
        category,
        items,
        ROUND(100.0 * items / SUM(items) OVER (PARTITION BY customer_state), 1) AS pct_of_state
    FROM per_state_category
    WHERE customer_state = 'RJ'
    ORDER BY items DESC
""")

,customer_state,category,items,pct_of_state
0,RJ,bed_bath_table,1644,11.3
1,RJ,furniture_decor,1090,7.5
2,RJ,health_beauty,1064,7.3
3,RJ,sports_leisure,1041,7.1
4,RJ,computers_accessories,1002,6.9
...,...,...,...,...
64,RJ,diapers_and_hygiene,2,0.0
65,RJ,fashio_female_clothing,2,0.0
66,RJ,cds_dvds_musicals,2,0.0
67,RJ,fashion_sport,2,0.0


In [11]:
query("""
      -- ex:a.1
    SELECT
        order_status,
        COUNT(*) AS orders_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_all
    FROM orders
    GROUP BY order_status
    ORDER BY orders_count DESC
""")

,order_status,orders_count,pct_of_all
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


In [12]:
query("""
      --ex:b.1
    WITH state_orders AS (
        SELECT
            c.customer_state,
            COUNT(*) AS orders_count
        FROM orders AS o
        JOIN customers AS c ON c.customer_id = o.customer_id
        GROUP BY c.customer_state
    ),
    with_avg AS (
        SELECT
            customer_state,
            orders_count,
            AVG(orders_count) OVER () AS avg_orders_per_state
        FROM state_orders
    )
    SELECT *
    FROM with_avg
    WHERE orders_count > avg_orders_per_state
    ORDER BY orders_count DESC
""")

,customer_state,orders_count,avg_orders_per_state
0,SP,41746,3683.0
1,RJ,12852,3683.0
2,MG,11635,3683.0
3,RS,5466,3683.0
4,PR,5045,3683.0


In [13]:
query("""
    SELECT
        seller_state,
        COUNT(*) AS sellers,
        ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS row_number,
        RANK()       OVER (ORDER BY COUNT(*) DESC) AS rank,
        DENSE_RANK() OVER (ORDER BY COUNT(*) DESC) AS dense_rank
    FROM sellers
    GROUP BY seller_state
    ORDER BY sellers DESC
""")

,seller_state,sellers,row_number,rank,dense_rank
0,SP,1849,1,1,1
1,PR,349,2,2,2
2,MG,244,3,3,3
3,SC,190,4,4,4
4,RJ,171,5,5,5
5,RS,129,6,6,6
6,GO,40,7,7,7
7,DF,30,8,8,8
8,ES,23,9,9,9
9,BA,19,10,10,10


In [14]:
per_state_category = """
    per_state_category AS (
        SELECT
            c.customer_state,
            COALESCE(t.product_category_name_english, 'unknown') AS category,
            COUNT(*) AS items
        FROM order_items AS i
        JOIN orders AS o ON o.order_id = i.order_id
        JOIN customers AS c ON c.customer_id = o.customer_id
        JOIN products AS p ON p.product_id = i.product_id
        LEFT JOIN category_translation AS t ON t.product_category_name = p.product_category_name
        GROUP BY c.customer_state, category
    )
"""

query(f"""
    WITH {per_state_category}
    SELECT
        category,
        items,
        ROW_NUMBER() OVER (ORDER BY items DESC, category) AS row_num,
        RANK()       OVER (ORDER BY items DESC) AS rnk,
        DENSE_RANK() OVER (ORDER BY items DESC) AS dense_rnk
    FROM per_state_category
    WHERE customer_state = 'AC'
""")

,category,items,row_num,rnk,dense_rnk
0,furniture_decor,12,1,1,1
1,computers_accessories,9,2,2,2
2,sports_leisure,9,3,2,2
3,health_beauty,7,4,4,3
4,telephony,5,5,5,4
5,auto,4,6,6,5
6,bed_bath_table,4,7,6,5
7,electronics,4,8,6,5
8,housewares,4,9,6,5
9,watches_gifts,4,10,6,5


In [15]:
query("""
    WITH first_sales AS (
        SELECT
            p.product_category_name,
            o.order_purchase_timestamp,
            o.order_id,
            ROW_NUMBER() OVER (PARTITION BY p.product_category_name ORDER BY o.order_purchase_timestamp, o.order_id) AS sale_num
        FROM order_items AS i
        JOIN orders AS o ON o.order_id = i.order_id
        JOIN products AS p ON p.product_id = i.product_id
    )
    SELECT
        COALESCE(t.product_category_name_english, 'unknown') AS category,
        order_purchase_timestamp AS first_sale_time,
        order_id
    FROM first_sales AS f
    LEFT JOIN category_translation AS t ON t.product_category_name = f.product_category_name
    WHERE sale_num = 1
    ORDER BY first_sale_time DESC
    LIMIT 5
""")

,category,first_sale_time,order_id
0,unknown,2018-05-08 17:18:14,7faf8b9448b66f4a3ac98a312b3e7609
1,flowers,2017-11-06 17:29:19,0b4e6da2eae282f0a0072085c7adacfe
2,party_supplies,2017-07-29 20:01:05,6914acb0f52f56529563f4f97062b90b
3,construction_tools_lights,2017-07-25 15:02:02,f3cb950f0d57842dbbd55317ceaeaa71
4,cds_dvds_musicals,2017-07-05 09:16:49,37104fd8b73d0691fda9491fe8906808


In [16]:
query(f"""
    WITH {per_state_category},
    ranked_categories AS (
        SELECT
            customer_state,
            category,
            items,
            RANK() OVER (PARTITION BY customer_state ORDER BY items DESC) AS place
        FROM per_state_category
    )
    SELECT
        category,
        COUNT(customer_state) AS states_where_first
    FROM ranked_categories
    WHERE place = 1
    GROUP BY category
    ORDER BY states_where_first DESC
""")

,category,states_where_first
0,health_beauty,16
1,bed_bath_table,6
2,sports_leisure,3
3,furniture_decor,2


In [17]:
   seller_orders = """
       seller_orders AS (
           -- one row = one order of one seller
           SELECT i.seller_id, o.order_id, o.order_purchase_timestamp
           FROM order_items AS i
           JOIN orders AS o ON o.order_id = i.order_id
           GROUP BY i.seller_id, o.order_id, o.order_purchase_timestamp
       )
   """

In [18]:
   query(f"""
       WITH {seller_orders},
       numbered AS (
           -- one row = one sale, with its number among the sales of its seller
           SELECT seller_id, order_id, order_purchase_timestamp,
                  ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY order_purchase_timestamp, order_id) AS sale_number
           FROM seller_orders
       )
       SELECT date_trunc('month', order_purchase_timestamp)::date AS month, COUNT(*) AS new_sellers
       FROM numbered
       WHERE sale_number = 1
         AND order_purchase_timestamp >= '2017-01-01'
         AND order_purchase_timestamp <  '2018-01-01'
       GROUP BY month
       ORDER BY month
   """)

,month,new_sellers
0,2017-01-01,151
1,2017-02-01,228
2,2017-03-01,173
3,2017-04-01,116
4,2017-05-01,124
5,2017-06-01,76
6,2017-07-01,115
7,2017-08-01,129
8,2017-09-01,127
9,2017-10-01,147


In [19]:
   query("""
       WITH weeks AS (
           -- one row = one week
           SELECT date_trunc('week', order_purchase_timestamp)::date AS week_start, COUNT(*) AS orders
           FROM orders
           WHERE order_purchase_timestamp >= '2017-10-02'
             AND order_purchase_timestamp <  '2018-01-01'
           GROUP BY week_start
       ),
       with_previous AS (
           SELECT week_start, orders, LAG(orders) OVER (ORDER BY week_start) AS previous_week
           FROM weeks
       )
       SELECT
           week_start,
           orders,
           previous_week,
           orders - previous_week AS change,
           ROUND(100.0 * (orders - previous_week) / previous_week, 1) AS pct_change
       FROM with_previous
       ORDER BY week_start
   """)

,week_start,orders,previous_week,change,pct_change
0,2017-10-02,999,NaN,NaN,NaN
1,2017-10-09,1070,999.0,71.0,7.1
2,2017-10-16,1147,1070.0,77.0,7.2
3,2017-10-23,978,1147.0,-169.0,-14.7
4,2017-10-30,942,978.0,-36.0,-3.7
5,2017-11-06,1217,942.0,275.0,29.2
6,2017-11-13,1313,1217.0,96.0,7.9
7,2017-11-20,3008,1313.0,1695.0,129.1
8,2017-11-27,2098,3008.0,-910.0,-30.3
9,2017-12-04,1692,2098.0,-406.0,-19.4


In [20]:
   query(f"""
       WITH {seller_orders},
       with_previous AS (
           SELECT seller_id, order_purchase_timestamp,
                  LAG(order_purchase_timestamp) OVER (PARTITION BY seller_id ORDER BY order_purchase_timestamp, order_id) AS previous_sale
           FROM seller_orders
       ),
       gaps AS (
           SELECT seller_id, previous_sale, EXTRACT(DAY FROM order_purchase_timestamp - previous_sale) AS days_since_previous
           FROM with_previous
       )
       SELECT
           COUNT(*) AS sales,
           COUNT(previous_sale) AS sales_with_a_previous_sale,
           ROUND(AVG(days_since_previous), 1) AS average_days,
           PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY days_since_previous) AS median_days
       FROM gaps
   """)

,sales,sales_with_a_previous_sale,average_days,median_days
0,100010,96915,5.2,1.0


In [23]:
query("""
    --ex:b.3
    WITH weeks AS (
        SELECT date_trunc('week', order_purchase_timestamp)::date AS week_start, COUNT(*) AS orders
        FROM orders
        WHERE order_purchase_timestamp >= '2017-01-01'
          AND order_purchase_timestamp <  '2018-01-01'
        GROUP BY week_start
    ),
    with_previous AS (
        SELECT week_start, orders, LAG(orders) OVER (ORDER BY week_start) AS previous_week
        FROM weeks
    )
    SELECT
        week_start,
        orders,
        previous_week,
        orders - previous_week AS change
    FROM with_previous
    ORDER BY change ASC
    LIMIT 1
""") 

,week_start,orders,previous_week,change
0,2017-11-27,2098,3008,-910


In [25]:
query("""
    WITH days AS (
        -- one row = one day
        SELECT order_purchase_timestamp::date AS day, COUNT(*) AS orders
        FROM orders
        WHERE order_purchase_timestamp >= '2017-11-01'
          AND order_purchase_timestamp <  '2017-12-01'
        GROUP BY day
    )
    SELECT day, orders, SUM(orders) OVER (ORDER BY day) AS orders_so_far
    FROM days
    ORDER BY day
""")

,day,orders,orders_so_far
0,2017-11-01,111,111.0
1,2017-11-02,124,235.0
2,2017-11-03,143,378.0
3,2017-11-04,111,489.0
4,2017-11-05,144,633.0
5,2017-11-06,193,826.0
6,2017-11-07,160,986.0
7,2017-11-08,175,1161.0
8,2017-11-09,191,1352.0
9,2017-11-10,165,1517.0


In [26]:
query("""
    WITH days AS (
        -- one row = one day
        SELECT order_purchase_timestamp::date AS day, COUNT(*) AS orders
        FROM orders
        WHERE order_purchase_timestamp >= '2017-10-01'
          AND order_purchase_timestamp <  '2018-01-01'
        GROUP BY day
    )
    SELECT
        day,
        orders,
        ROUND(AVG(orders) OVER (ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 1) AS average_last_7_days
    FROM days
    ORDER BY day
""")

,day,orders,average_last_7_days
0,2017-10-01,128,128.0
1,2017-10-02,143,135.5
2,2017-10-03,198,156.3
3,2017-10-04,157,156.5
4,2017-10-05,140,153.2
...,...,...,...
87,2017-12-27,167,121.0
88,2017-12-28,146,121.4
89,2017-12-29,135,124.9
90,2017-12-30,97,123.1


In [27]:
 query("""
       SELECT
           order_id,
           order_purchase_timestamp,
           COUNT(*) OVER (ORDER BY order_purchase_timestamp::date) AS orders_so_far_by_date,
           COUNT(*) OVER (ORDER BY order_purchase_timestamp, order_id) AS orders_so_far_by_time
       FROM orders
       WHERE order_purchase_timestamp >= '2017-11-01'
         AND order_purchase_timestamp <  '2017-12-01'
       ORDER BY order_purchase_timestamp, order_id
       LIMIT 5
   """)

,order_id,order_purchase_timestamp,orders_so_far_by_date,orders_so_far_by_time
0,9808a7f5f5446817c3ebf7a804111d53,2017-11-01 00:12:34,111,1
1,c1055c3c83d15a7d43c7b6e3f283ae53,2017-11-01 00:24:57,111,2
2,ed130013de4053d7678825d4d8e7deb1,2017-11-01 00:43:33,111,3
3,08b450b388221bfe5472ed23887a7564,2017-11-01 01:48:36,111,4
4,af29f3d2878958723720a759676814e7,2017-11-01 01:52:21,111,5


In [ ]:
query("""
      --ex : a.4
    SELECT COUNT(DISTINCT order_purchase_timestamp::date) AS active_days
    FROM orders
    WHERE order_purchase_timestamp >= '2017-10-01'
      AND order_purchase_timestamp <  '2018-01-01'
""")

,active_days
0,92


In [30]:
query(f"""
      --ex:b.4
    WITH {seller_orders},
    numbered AS (
        SELECT seller_id, order_id, order_purchase_timestamp,
               ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY order_purchase_timestamp, order_id) AS sale_number
        FROM seller_orders
    ),
    new_sellers_per_month AS (
        SELECT date_trunc('month', order_purchase_timestamp)::date AS month, COUNT(*) AS new_sellers
        FROM numbered
        WHERE sale_number = 1
          AND order_purchase_timestamp >= '2017-01-01'
          AND order_purchase_timestamp <  '2018-01-01'
        GROUP BY month
    )
    SELECT
        month,
        new_sellers,
        SUM(new_sellers) OVER (ORDER BY month) AS cumulative_new_sellers
    FROM new_sellers_per_month
    ORDER BY month
""")

,month,new_sellers,cumulative_new_sellers
0,2017-01-01,151,151.0
1,2017-02-01,228,379.0
2,2017-03-01,173,552.0
3,2017-04-01,116,668.0
4,2017-05-01,124,792.0
5,2017-06-01,76,868.0
6,2017-07-01,115,983.0
7,2017-08-01,129,1112.0
8,2017-09-01,127,1239.0
9,2017-10-01,147,1386.0


In [31]:
query("""
    WITH days AS (
        SELECT order_purchase_timestamp::date AS day, COUNT(*) AS orders
        FROM orders
        WHERE order_purchase_timestamp >= '2017-10-01'
          AND order_purchase_timestamp <  '2018-01-01'
        GROUP BY day
    )
    SELECT
        day,
        orders,
        ROUND(AVG(orders) OVER (ORDER BY day ROWS BETWEEN 3 PRECEDING AND 3 FOLLOWING), 1) AS centered_avg_7_days
    FROM days
    ORDER BY day
""")


,day,orders,centered_avg_7_days
0,2017-10-01,128,156.5
1,2017-10-02,143,153.2
2,2017-10-03,198,149.3
3,2017-10-04,157,143.0
4,2017-10-05,140,142.7
...,...,...,...
87,2017-12-27,167,123.1
88,2017-12-28,146,125.3
89,2017-12-29,135,131.2
90,2017-12-30,97,123.8
